In [1]:

import pandas as pd
import re
import ast
from html import escape
from IPython.display import display, HTML

In [2]:
df = pd.read_csv("vergaderstukken_rijksoverheid_ai_related.csv", index_col=0)
df.head()

,title,introduction,canonical,dataurl,frontenddate,lastmodified,available,ai_related,company_hits,pdf_text,relevant_text,matched_keywords,type
id,,,,,,,,,,,,,
bb359e45-fd49-43fb-9146-f4c1bf57c632,Geannoteerde besluitenlijst ministerraad 28 me...,<p>Overzicht van alle besluiten die de ministe...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2021-05-28T16:38:00.000Z,2022-07-04T14:45:15.697Z,2021-05-28T16:38:00.000Z,yes,[],MINISTERRAAD\nKenmerk : 4206864\nBESLUITENLIJS...,Conclusies van de coördinatiecommissie d.d. 25...,['kunstmatige intelligentie'],vergaderstuk
bed99db0-7aeb-419e-8a86-2cc51879c97e,Agenda ministerraad 4 juni 2021,<p>De agenda toont de onderwerpen die in de mi...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2021-06-04T10:11:00.000Z,2022-07-04T14:38:13.818Z,2021-06-04T10:11:00.000Z,yes,[],MINISTERRAAD\nKenmerk : 3753052\nAGENDA\nVerga...,Programma Landelijke Vreemdelingen Voorziening...,"['algoritmen', 'artificiële intelligentie']",vergaderstuk
51a0771c-c7a2-40a7-8520-f474511e2e2f,Geannoteerde besluitenlijst ministerraad 4 jun...,<p>Overzicht van alle besluiten die de ministe...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2021-06-04T15:07:00.000Z,2022-07-04T14:33:23.791Z,2021-06-04T15:07:00.000Z,yes,[],MINISTERRAAD\nKenmerk : 4208548\nBESLUITENLIJS...,"1 juni 2021,\nnr.22 (Minister van BZ)\nDe conc...","['ai', 'algoritmen', 'artificiële intelligenti...",vergaderstuk
f832f415-fda2-47b5-be67-648187989570,Geannoteerde besluitenlijst ministerraad 29 ok...,<p>Overzicht van alle besluiten die de ministe...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2021-10-29T17:05:00.000Z,2022-07-01T11:48:39.669Z,2021-10-29T17:05:00.000Z,yes,['x'],MINISTERRAAD\nKenmerk : 4232087\nBESLUITENLIJS...,4. EU-implementatie\na. Wijziging van het Alge...,"['x', 'bard']",vergaderstuk
ef0f22b8-5d46-4bb8-9627-9b9b615ba235,Geannoteerde besluitenlijst ministerraad 26 no...,<p>Overzicht van alle besluiten die de ministe...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2021-11-26T16:30:00.000Z,2022-07-01T07:59:10.267Z,2021-11-26T16:30:00.000Z,yes,[],MINISTERRAAD\nKenmerk : 4237648\nBESLUITENLIJS...,Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,"['ai', 'artificial intelligence']",vergaderstuk


In [7]:
import ast

def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

df['matched_keywords'] = df.apply(
    lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords']),
    axis=1
)
df['matched_keywords'].iloc[0]

['kunstmatige intelligentie']

In [5]:


def inspect_ai_related(df, body='relevant_text',  num_samples=2, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """
    
    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', body}.issubset(df.columns):
        missing = {'title', body} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

  
    

    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        
        ai_val = row['ai_related']
        title = row['title']
        body_text = row[body]  # ✅ 'body' parameter stays intact

         # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples
    

In [8]:
# program = "all" to display from all programs, or specify a program like "jinek"
inspect_ai_related(df, body = 'relevant_text', num_samples=2, random_state=42)

Displayed 2 random articles (with row-specific matched keywords highlighted).


,title,introduction,canonical,dataurl,frontenddate,lastmodified,available,ai_related,company_hits,pdf_text,relevant_text,matched_keywords,type
id,,,,,,,,,,,,,
cf056dd8-ec45-477f-960e-91957a04ab06,Ministerie van Financiën Audit Committee stukk...,<p>Dit zijn de stukken die het Audit Committee...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2025-10-15T00:00:00.000Z,2025-12-22T14:45:00.917Z,2025-12-22T14:41:32.808Z,yes,[],Doc nr. Datum Titel doc.\n1 15-10-2025 AC 253....,Pagina 2van 2\n0001\nActiepuntenlijst Audit Co...,"[ai, algoritme, algoritmen]",vergaderstuk
66c00e97-f239-444b-a4e1-1bee0f988529,Ministerie van Financiën Audit Committee stukk...,<p>Dit zijn de stukken die het Audit Committee...,https://www.rijksoverheid.nl/documenten/vergad...,https://opendata.rijksoverheid.nl/v1/documents...,2025-12-03T00:00:00.000Z,2026-03-17T08:35:00.831Z,2026-03-17T08:28:55.603Z,yes,[],Doc nr. Datum Titel doc.\n1 11-11-2025 AC 254....,In een later stadium zal het\nauditplan door F...,[ai],vergaderstuk
